# Drift-aware budgeted FL — Colab launcher

Thin launcher: all logic lives in `src/`. Run cells top to bottom. CPU runtime is enough (the MLP is tiny); a GPU is not used.

**Order:** clone → install → tests → download UCI data → clinical calibration (dev only) → clinical runs → analysis.

In [ ]:
# 1. Optional: keep results on Google Drive so a disconnect doesn't lose runs
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/drift-aware-fl-runs'
else:
    OUT = '/content/drift-aware-fl/results/runs'
print('run outputs ->', OUT)

In [ ]:
# 2. Clone (or update) the repository
%cd /content
!test -d drift-aware-fl || git clone https://github.com/DhanviMohanraj/drift-aware-FL.git drift-aware-fl
%cd /content/drift-aware-fl
!git pull --ff-only
!git log --oneline -3

In [ ]:
# 3. Install dependencies (torch is preinstalled on Colab)
!pip install -q -r requirements.txt
import torch, sklearn, pandas; print(torch.__version__, sklearn.__version__, pandas.__version__)

In [ ]:
# 4. Correctness gates — every test must pass before experiments
!python -m pytest -q

In [ ]:
# 5. Download UCI Diabetes 130-US Hospitals (id 296) + write data/raw/data_manifest.json (sha256, URL, date)
!python -m src.data --out data/raw
!cat data/raw/data_manifest.json

In [ ]:
# 6. Clinical calibration on DEVELOPMENT patients only (threshold from {3,6,12}, periodic interval from {2,5,10})
!python -m src.calibrate --dataset clinical

In [ ]:
# 7. Freeze the clinical calibration into configs/core.yaml (dataset_overrides.clinical) — do this ONCE, before final runs
import json, yaml
cal = json.load(open('results/calibration_clinical.json'))
cfg = yaml.safe_load(open('configs/core.yaml'))
cfg['dataset_overrides']['clinical'] = {'detector': {'threshold': cal['selected_threshold']},
                                        'policy': {'periodic_interval': cal['selected_periodic_interval']}}
yaml.safe_dump(cfg, open('configs/core.yaml', 'w'), sort_keys=False)
print(cfg['dataset_overrides'])

In [ ]:
# 8. Profile: one clinical run per method on C1 (seed 11) — check runtime before launching tiers
!python -m src.matrix --tier minimum --only-dataset clinical --dry-run --out "$OUT"
!python -m src.run --dataset clinical --scenario C1 --method always_on --seed 11 --out "$OUT"

In [ ]:
# 9. Clinical tiers in priority order (re-running skips completed runs; safe after a disconnect)
!python -m src.matrix --tier minimum  --only-dataset clinical --workers 2 --out "$OUT"
!python -m src.matrix --tier full     --only-dataset clinical --workers 2 --out "$OUT"
!python -m src.matrix --tier ablation --only-dataset clinical --workers 2 --out "$OUT"
!python -m src.matrix --tier sweep    --only-dataset clinical --workers 2 --out "$OUT"

In [ ]:
# 10. (Optional) re-run the synthetic tiers yourself instead of using the committed results
# !python -m src.matrix --tier full --only-dataset synthetic --workers 2 --out "$OUT"
# !python -m src.matrix --tier ablation --only-dataset synthetic --workers 2 --out "$OUT"
# !python -m src.matrix --tier sweep --only-dataset synthetic --workers 2 --out "$OUT"

In [ ]:
# 11. Analysis from declared manifests only
!python -m src.analyze --manifests results/manifest_full_synthetic.json results/manifest_ablation_synthetic.json results/manifest_sweep_synthetic.json results/manifest_full_clinical.json results/manifest_ablation_clinical.json results/manifest_sweep_clinical.json --out results/combined
from IPython.display import Markdown, Image, display
display(Markdown(open('results/combined/tables/results.md').read()))

In [ ]:
import glob
for f in sorted(glob.glob('results/combined/figures/*C1*.png') + glob.glob('results/combined/figures/*S3*.png')):
    print(f); display(Image(f))